# 02 - Final Evaluation and Model Comparison

This notebook summarizes final evaluation artifacts generated by `src/evaluate.py` and `src/interpret.py`.

It is intentionally report-focused:
- load canonical metrics JSON files,
- compare model metrics side-by-side,
- review confusion matrices and training curves,
- highlight pneumonia recall (sensitivity) as the primary screening metric.

All metrics and figures are read from `results/` outputs (no duplicate training/evaluation logic in notebook cells).

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS = ROOT / "results"
METRICS_DIR = RESULTS / "metrics"
PLOTS_DIR = RESULTS / "plots"
CM_DIR = RESULTS / "confusion_matrices"

metric_paths = {
    "custom_cnn": METRICS_DIR / "custom_cnn_metrics.json",
    "densenet121": METRICS_DIR / "densenet121_metrics.json",
}

for name, path in metric_paths.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing metrics file for {name}: {path}")

print("Using results directory:", RESULTS)
print("Metrics files:")
for name, path in metric_paths.items():
    print(f"  - {name}: {path}")

In [ ]:
def load_metrics(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))

rows = []
for model_name, path in metric_paths.items():
    payload = load_metrics(path)
    report = payload["classification_report"]
    rows.append(
        {
            "model": model_name,
            "accuracy": payload["accuracy"],
            "precision_normal": report["NORMAL"]["precision"],
            "recall_normal": report["NORMAL"]["recall"],
            "f1_normal": report["NORMAL"]["f1-score"],
            "precision_pneumonia": report["PNEUMONIA"]["precision"],
            "recall_pneumonia": report["PNEUMONIA"]["recall"],
            "f1_pneumonia": report["PNEUMONIA"]["f1-score"],
            "macro_f1": report["macro avg"]["f1-score"],
            "weighted_f1": report["weighted avg"]["f1-score"],
            "num_samples": payload["num_samples"],
            "checkpoint_epoch_meta": payload.get("checkpoint_metadata", {}).get("epoch"),
        }
    )

comparison = pd.DataFrame(rows).sort_values("model").reset_index(drop=True)

display(Markdown("## Final Metric Comparison"))
display(comparison)

display(Markdown("### Clinical Priority: Pneumonia Recall (Sensitivity)"))
for _, row in comparison.iterrows():
    display(Markdown(f"- **{row['model']}** pneumonia recall: **{row['recall_pneumonia']:.4f}**"))

In [ ]:
display(Markdown("## Confusion Matrices"))
for model_name in ["custom_cnn", "densenet121"]:
    img_path = CM_DIR / f"{model_name}_confusion_matrix.png"
    display(Markdown(f"### {model_name}"))
    if img_path.exists():
        display(Image(filename=str(img_path)))
    else:
        print("Missing:", img_path)

display(Markdown("## Training Curves"))
for img_name in [
    "custom_cnn_training_curves.png",
    "densenet121_training_curves.png",
    "model_comparison_validation_curves.png",
]:
    img_path = PLOTS_DIR / img_name
    display(Markdown(f"### {img_name}"))
    if img_path.exists():
        display(Image(filename=str(img_path)))
    else:
        print("Missing:", img_path)

display(Markdown("## Interpretation Notes"))
display(
    Markdown(
        "\n".join([
            "- Both evaluated models show identical behavior in the current checkpoint set.",
            "- Pneumonia recall is high (1.0), but NORMAL recall is 0.0, meaning all samples were predicted as PNEUMONIA.",
            "- This operating point is unsuitable for balanced diagnostic performance and requires checkpoint provenance review and/or retraining calibration."
        ])
    )
)